In [1]:
import tensorflow as td
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

/Users/muralidharsuddhapalli/Documents/Python_ML/DL_ANN_project_01/venv/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
import sklearn
import threadpoolctl
import joblib

print("sklearn:", sklearn.__version__)
print("threadpoolctl:", threadpoolctl.__version__)
print("joblib:", joblib.__version__)

sklearn: 1.9.0
threadpoolctl: 3.6.0
joblib: 1.5.3


In [25]:
### Load the ANN trained model, scaler pickle, onehot
model = load_model('model.h5')

## load the encoder and scaler
with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [36]:
## Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender' : 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance':60000,
    'NumOfProducts':2,
    'HasCrCard':1,
    'IsActiveMember':1,
    'EstimatedSalary':50000
}

In [16]:
import pandas as pd
from sklearn.preprocessing import StandardScaler,LabelEncoder

In [37]:
# One-hot encoding of 'Geography'
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

/Users/muralidharsuddhapalli/Documents/Python_ML/DL_ANN_project_01/venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [38]:
input_data_df = pd.DataFrame([input_data])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [39]:
## Encode the categorical variables
input_data_df['Gender'] = label_encoder_gender.transform(input_data_df['Gender'])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [40]:
## Concatenation with on hot encoded
input_data_df = pd.concat([input_data_df.drop("Geography", axis = 1), geo_encoded_df], axis =1)

In [41]:
input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [44]:
## Scaling the data
input_scaled = scaler.transform(input_data_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [45]:
## Prediction
prediction = model.predict(input_scaled)

1/1 [==============================] - 0s 136ms/step


In [46]:
prediction

array([[0.02184642]], dtype=float32)

In [48]:
prediction_proba = prediction[0][0]
prediction_proba

0.021846421

In [49]:
if(prediction_proba>0.5):
    print("The customer is likely to churn")
else:
    print('The customer is not likely to churn')

The customer is not likely to churn
